In [1]:
from collections import deque
import random
import numpy as np
from player import Player
from one_round import OneRound
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import torch

In [2]:
class ReplayBuffer:
    def __init__(self,buffer_size,batch_size):
        self.buffer = deque(maxlen = buffer_size)
        self.batch_size = batch_size
    
    def add(self,state,action,reward,next_state,done):
        data = (state,action,reward,next_state,done)
        self.buffer.append(data)
    
    def __len__(self):
        return len(self.buffer)
    
    def get_batch(self):
        data = random.sample(self.buffer,self.batch_size)

        state = np.stack([x[0] for x in data])  # リスト内包表記で明示的にリストを作成
        action = np.stack([x[1] for x in data])
        reward = np.stack([x[2] for x in data])
        next_state = np.stack([x[3] for x in data])
        done = np.stack([x[4] for x in data])

        state = torch.tensor(state, dtype=torch.float32)
        action = torch.tensor(action, dtype=torch.int64)  # アクションは整数型
        reward = torch.tensor(reward, dtype=torch.float32)
        next_state = torch.tensor(next_state, dtype=torch.float32)
        done = torch.tensor(done, dtype=torch.float32)

        return state,action,reward,next_state,done

In [3]:
class QNet(nn.Module):
    def __init__(self, action_size):
        super().__init__()
        self.l1 = nn.Linear(11,128)
        self.l2 = nn.Linear(128,128)
        self.l3 = nn.Linear(128,action_size)

    def forward(self,x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = self.l3(x)
        return x

In [4]:
class DQNAgent:
    def __init__(self):
        self.gamma = 0.98
        self.lr = 0.00005
        self.epsilon = 0.1
        self.buffer_size = 10000
        self.batch_size = 32
        self.action_size = 3

        self.replay_buffer = ReplayBuffer(self.buffer_size,self.batch_size)
        self.qnet = QNet(self.action_size)
        self.qnet_target = QNet(self.action_size)
        self.optimizer = optim.Adam(self.qnet.parameters(), lr=self.lr)

    def sync_qnet(self):
        self.qnet_target.load_state_dict(self.qnet.state_dict())

    def get_action(self,state,mask):
        state = torch.tensor(state[np.newaxis, :], dtype=torch.float32)
        mask = torch.tensor(mask,dtype=torch.float32)
        if np.random.rand() < self.epsilon:
            valid_actions = torch.nonzero(mask).squeeze(-1).numpy()
            return np.random.choice(valid_actions)
        else:
            qs = self.qnet(state)
            qs_mask = qs * mask
            return qs_mask.argmax().item()

    def update(self, state, action, reward, next_state, done):

        def preprocess_next_state(next_state, target_shape):
            if next_state is None:
                return np.zeros(target_shape)  # ダミーの状態
            return next_state
        
        ###注意！！！！（magic number）
        next_state = preprocess_next_state(next_state,6)

        state = torch.tensor(state, dtype=torch.float32)
        action = torch.tensor(action, dtype=torch.int64)  # アクションは整数型
        reward = torch.tensor(reward, dtype=torch.float32)
        next_state = torch.tensor(next_state, dtype=torch.float32)
        done = torch.tensor(done, dtype=torch.float32)

        self.replay_buffer.add(state, action, reward, next_state, done)
        if len(self.replay_buffer) < self.batch_size:
            return

        state, action, reward, next_state, done = self.replay_buffer.get_batch()
        qs = self.qnet(state)### qs = 各アクションのq値
        q = qs[np.arange(len(action)), action]### q = 実際に行なったアクションのq値

        next_qs = self.qnet_target(next_state)
        next_q = next_qs.max(1)[0]

        next_q.detach()
        target = reward + (1 - done) * self.gamma * next_q

        loss_fn = nn.MSELoss()
        loss = loss_fn(q, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def copy_from(self, other_agent):
        # other_agent からパラメータをコピー
        self.qnet.load_state_dict(other_agent.qnet.state_dict())
        self.optimizer.load_state_dict(other_agent.optimizer.state_dict())

In [ ]:
episodes = 1000

sync_interval = 20
agent = DQNAgent()
agent_sb = DQNAgent()

reward_history = []
count_f = 0
count_c = 0
count_r = 0

for episode in range(episodes):
    print("epoch:",episode)

    ## ゲームの設定
    ## プレイヤーは二人
    players = []
    for i in range(2):
        players.append(Player(100000,"player" + str(i),i))

    one_round = OneRound(players,0,100)
    one_round.set()

    ## ゲームの初期条件を入手
    total_reward = 0
    player_index = one_round.current_index
    state_a = one_round.player_state(player_index)

    while True:
        mask_a = one_round.mask(one_round.current_index)
        action_a = agent.get_action(state_a,mask_a)

        if action_a == 0:
            action_a_decord = "f"
            count_f += 1
        elif action_a == 1:
            action_a_decord = "c"
            count_c += 1
        elif action_a == 2:
            action_a_decord = "r"
            count_r += 1

        reward_a, state_b = one_round.step(action_a_decord)

        if action_a_decord == "f":
            agent.update(state_a,action_a,reward_a,None,True)
            break
        if one_round.current_phase == 4:
            agent.update(state_a,action_a,reward_a,None,True)
            break

        mask_b = one_round.mask(one_round.current_index)
        action_b = agent_sb.get_action(state_b,mask_b)

        if action_b == 0:
            action_b_decord = "f"
            count_f += 1
        if action_b == 1:
            action_b_decord = "c"
            count_c += 1
        if action_b == 2:
            action_b_decord = "r"
            count_r += 1

        reward_b, next_state_a = one_round.step(action_b_decord)

        if action_b_decord == "f":
            agent.update(state_a,action_a,reward_a,None,True)
            break
        if one_round.current_phase == 4:
            agent.update(state_a,action_a,reward_a,None,True)
            break

        agent.update(state_a, action_a, reward_a, next_state_a, False)
        state_a = next_state_a
        total_reward += reward_a

    if episode % sync_interval == 0:
        agent.sync_qnet()
        agent_sb.copy_from(agent)

    reward_history.append(total_reward)

In [ ]:
import matplotlib.pyplot as plt


data = np.array(reward_history)
group_size = 50
averaged_data = [np.mean(data[i:i+group_size]) for i in range(0, len(data), group_size)]

# グラフの描画
plt.figure(figsize=(10, 6))  # グラフのサイズを指定
plt.plot(averaged_data, label='Reward per Episode', marker='o')

# グラフの装飾
plt.title("Reward History per Episode", fontsize=16)  # タイトル
plt.xlabel("Episode", fontsize=14)  # x軸ラベル
plt.ylabel("Total Reward", fontsize=14)  # y軸ラベル
plt.grid(True)  # グリッド表示